In [1]:
from agents import Agent, Runner, AsyncOpenAI, OpenAIChatCompletionsModel
from agents.mcp import (
    MCPServerStreamableHttp,
    MCPServerStreamableHttpParams,
    MCPServerStdio,
    MCPServerStdioParams,
    MCPServerSse,
    MCPServerSseParams,
)
from agents.model_settings import ModelSettings
from agents.extensions.visualization import draw_graph
from agents import (
    set_default_openai_client,
    set_default_openai_key,
    set_tracing_disabled,
)
import asyncio
import os


set_default_openai_key(os.getenv("LLM_API_KEY"), False)
set_tracing_disabled(True)

In [2]:
openai_like_client = AsyncOpenAI(
    api_key=os.getenv("LLM_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    
)

model = OpenAIChatCompletionsModel(
    model="gemini-2.5-flash-lite", openai_client=openai_like_client
)

postgres_server = MCPServerSse(MCPServerSseParams(url="http://localhost:8002/sse"))

In [3]:
final_response_agent = Agent(
    name="final_response_agent",
    instructions="You are an assistant that response to the user, using information from the previous agent to answer the query.",
    model=model,
)

postgres_agent = Agent(
    name="database_agent",
    instructions="You are in charge of a database related to my transactions. You must answer any questions related to transactions. You have tools to access a Postgres database, especially a table named 'transactions'.",
    model = model,
    mcp_servers=[postgres_server],
    handoffs=[final_response_agent],
)

rewrite_agent = Agent(
    name="rewrite_agent",
    instructions=f"""You are an assistant that refine the user query and delegate task to a sub-agent.""",
    model=model,
    handoffs=[postgres_agent],
    handoff_description="You should delegate tasks to at least one sub agent if need be.",
)

In [9]:
async with postgres_server:
    response = await Runner.run(
        starting_agent=rewrite_agent, input="Can you list out my 5 latest transactions?"
    )
    print(response.final_output)
    print(response.last_agent.name)

Here are your 5 latest transactions:

*   **ID:** 244, **Date:** 2025-07-07, **Detail:** MPOS*WALLACE,QUAN 5VIET NAM, **Amount:** 102000 VND, **Category:** Miscellaneous/Other
*   **ID:** 243, **Date:** 2025-07-07, **Detail:** SHEN HENG CHANG 00264,TAOYUAN CITYTAIWAN, **Amount:** 1035.00 TWD, **Category:** Entertainment & Lifestyle
*   **ID:** 242, **Date:** 2025-07-06, **Detail:** TOPS MARKET-AU CO,HO CHI MINHVIET NAM, **Amount:** 83600 VND, **Category:** Food & Dining
*   **ID:** 241, **Date:** 2025-07-06, **Detail:** WCM_WINMART_1705_VUNGTAU,BR VUNGTAUVIET NAM, **Amount:** 50012 VND, **Category:** Food & Dining
*   **ID:** 240, **Date:** 2025-07-06, **Detail:** PAYOO NTLONGCHAU80051,VUNGTAUVIET NAM, **Amount:** 47000 VND, **Category:** Personal Care & Health
database_agent
